In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
!pip install torch-geometric


In [ ]:
!pip -q install -U transformers peft bitsandbytes accelerate sentencepiece pandas


In [ ]:
from google.colab import drive
import os
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

base_drive_path = Path("/content/drive/MyDrive/Altegrad")

In [ ]:
import pickle
import torch

# 1. Load the data

TRAIN_GRAPHS = base_drive_path / 'train_graphs.pkl'

VAL_GRAPHS = base_drive_path / 'validation_graphs.pkl'

TEST_GRAPHS = base_drive_path / 'test_graphs.pkl'

with open(TRAIN_GRAPHS, 'rb') as f: train_graphs = pickle.load(f)
with open(VAL_GRAPHS, 'rb') as f:   val_graphs = pickle.load(f)
with open(TEST_GRAPHS, 'rb') as f:  test_graphs = pickle.load(f)


In [ ]:
from torch_geometric.nn import global_add_pool

def extract_global_features(x, batch_index):
    """
    Computes explicit features: Total Atoms + Counts of specific interesting atoms.
    Assumes x[:, 0] is Atomic Number.
    """
    # 1. Total Atoms per molecule
    num_atoms = torch.bincount(batch_index).float().unsqueeze(1)

    # 2. Define interesting atoms (Atomic Numbers)
    # C=6, N=7, O=8, F=9, P=15, S=16, Cl=17, Br=35, I=53
    # These cover 99% of organic chemistry.
    interesting_atoms = [6, 7, 8, 9, 15, 16, 17, 35, 53]

    atomic_nums = x[:, 0] # shape [Total_Nodes]

    counts_list = []

    # Track which atoms we have already counted to calculate "Others"
    counted_mask = torch.zeros_like(atomic_nums, dtype=torch.bool)

    for atom_num in interesting_atoms:
        # Create mask for this specific atom
        is_atom = (atomic_nums == atom_num).float()

        # Update the 'counted' mask (logic: current OR new)
        counted_mask = counted_mask | (atomic_nums == atom_num)

        # Sum per molecule
        count = global_add_pool(is_atom, batch_index).unsqueeze(1)
        counts_list.append(count)

    # 3. Count "Others" (Metals, rare elements, etc.)
    # Invert mask: True where atom was NOT in our list
    is_other = (~counted_mask).float()
    count_other = global_add_pool(is_other, batch_index).unsqueeze(1)

    # 4. Concatenate everything
    # Result: [Batch, 1 (Total) + 9 (Specific) + 1 (Other)] = 11 Features
    global_feats = torch.cat([num_atoms] + counts_list + [count_other], dim=1)

    # Log transform is crucial because counts can vary wildly (e.g. 1 vs 50 carbons)
    return torch.log1p(global_feats)

# Quick test

In [ ]:
from torch_geometric.loader import DataLoader
import torch

# 1. Instantiate a small temporary loader (Batch size 4 is enough to see)
test_loader = DataLoader(train_graphs, batch_size=4, shuffle=True)

# 2. Get the very first batch
batch = next(iter(test_loader))

# 3. Run the extractor
# Note: This returns log-transformed values (log(count + 1))
logged_features = extract_global_features(batch.x, batch.batch)

# 4. Convert back to counts so we can read them manually
# exp(x) - 1 reverses the log1p operation
real_counts = torch.expm1(logged_features)

# ==========================================
# 5. INSPECT THE RESULTS
# ==========================================
feature_names = ["Total", "C", "N", "O", "F", "P", "S", "Cl", "Br", "I", "Other"]

print(f"Batch contains {batch.num_graphs} molecules.\n")

for i in range(batch.num_graphs):
    print(f"--- Molecule {i+1} ---")

    # Get the row for this molecule
    row = real_counts[i]

    # Print non-zero counts
    description = []
    for name, count in zip(feature_names, row):
        if count > 0.1: # Threshold > 0.1 to avoid float errors
            description.append(f"{name}: {int(round(count.item()))}")

    print(f"  Computed Features: {', '.join(description)}")

    # OPTIONAL: Compare with the real text description if available
    # This helps you verify if the count makes sense structurally
    if hasattr(batch, 'description'):
        print(f"  Real Description:  {batch.description[i][:100]}...")
    print("")


# Training

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import autocast
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, GlobalAttention
from torch_geometric.utils import to_dense_batch
from transformers import T5Tokenizer, T5ForConditionalGeneration, get_cosine_schedule_with_warmup
from transformers.modeling_outputs import BaseModelOutput
from tqdm import tqdm
import os
import numpy as np

# ==========================================
# 0. CONFIGURATION & OPTIMIZATION
# ==========================================
# A100 Optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

CONFIG = {
    'batch_size': 64,
    'lr': 1e-5,
    'epochs': 30,
    'max_len': 128,
    'seed': 42,
    'save_path': base_drive_path / 'best_model_cl_finetuned.pth',
    'prev_model_path': base_drive_path / 'best_model_cl_finetuned.pth',
    'cl_dim': 256
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 1. HELPER CLASSES
# ==========================================
def extract_global_features(x, batch_index):
    """Simple global feature extraction placeholder."""
    num_graphs = batch_index.max().item() + 1
    # Taking first 11 dims or padding if x is smaller
    feats = x[:, :11].float()
    if feats.shape[1] < 11:
        padding = torch.zeros((feats.shape[0], 11 - feats.shape[1]), device=x.device)
        feats = torch.cat([feats, padding], dim=1)

    # Global mean pool manually
    global_feats = torch.zeros((num_graphs, 11), device=x.device)
    for i in range(num_graphs):
        mask = (batch_index == i)
        if mask.sum() > 0:
            global_feats[i] = feats[mask].mean(dim=0)
    return global_feats


class InfoNCELoss(nn.Module):
    # We explicitly add 'learnable' here so the error goes away
    def __init__(self, temperature=0.07, learnable=True):
        super().__init__()
        if learnable:
            self.temperature = nn.Parameter(torch.tensor(temperature))
        else:
            self.register_buffer('temperature', torch.tensor(temperature))

    def forward(self, graph_features, text_features):
        graph_features = F.normalize(graph_features, dim=-1)
        text_features = F.normalize(text_features, dim=-1)

        logits = torch.matmul(graph_features, text_features.T)

        # Clamp to avoid numerical instability
        temp = torch.clamp(self.temperature, min=1e-4, max=0.5)
        logits = logits / temp

        labels = torch.arange(logits.size(0), device=logits.device)

        loss_i2t = F.cross_entropy(logits, labels)
        loss_t2i = F.cross_entropy(logits.T, labels)

        return (loss_i2t + loss_t2i) / 2



# ==========================================
# Encoder Architecture
# ==========================================

class AtomEncoder(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        # 9 types of atomic features standard in OGB/Rdkit
        self.embeddings = nn.ModuleList([
            nn.Embedding(120, hidden_dim), # Atomic Num
            nn.Embedding(5, hidden_dim),   # Chirality
            nn.Embedding(12, hidden_dim),  # Degree
            nn.Embedding(12, hidden_dim),  # Formal Charge
            nn.Embedding(10, hidden_dim),  # Num H
            nn.Embedding(10, hidden_dim),  # Radical e-
            nn.Embedding(10, hidden_dim),  # Hybridization
            nn.Embedding(10, hidden_dim),  # Is Aromatic
            nn.Embedding(10, hidden_dim)   # Is in Ring
        ])

    def forward(self, x):
        out = 0
        # Sum embeddings for all available features
        num_feats = min(x.size(1), len(self.embeddings))
        for i in range(num_feats):
            out += self.embeddings[i](x[:, i].long())
        return out

class HybridGraphEncoder(nn.Module):
    """
    A 'Stronger' Encoder:
    1. Uses GATv2 for local chemical topology (bonds).
    2. Uses a Standard Transformer Encoder for global molecular reasoning.
    3. Aligns perfectly with T5's architecture.
    """
    def __init__(self, num_edge_features, gnn_hidden_dim=256, output_dim=768,
                 gnn_layers=2, transformer_layers=4, dropout=0.1,
                 num_global_feats=11, cl_embed_dim=256):
        super().__init__()

        self.gnn_hidden_dim = gnn_hidden_dim
        self.atom_encoder = AtomEncoder(gnn_hidden_dim)

        # --- 1. Local Structure Extraction (GNN) ---
        self.gnn_layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList()

        for _ in range(gnn_layers):
            self.gnn_layers.append(
                GATv2Conv(gnn_hidden_dim, gnn_hidden_dim, heads=4,
                          concat=False, dropout=dropout, edge_dim=num_edge_features)
            )
            self.norm_layers.append(nn.LayerNorm(gnn_hidden_dim))

        # --- 2. Global Reasoning (Transformer) ---
        # We process the graph as a dense sequence here [Batch, Max_Atoms, Dim]
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=gnn_hidden_dim,
            nhead=8,
            dim_feedforward=gnn_hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True # Pre-LN is better for stability
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=transformer_layers)

        # Learnable "CLS" token for graph summary
        self.cls_token = nn.Parameter(torch.randn(1, 1, gnn_hidden_dim))

        # --- 3. Projections ---
        # Project Transformer output to T5 dimension
        self.project_nodes = nn.Sequential(
            nn.Linear(gnn_hidden_dim, output_dim),
            nn.LayerNorm(output_dim)
        )

        # Contrastive Head (Project CLS token to CL dim)
        self.cl_projector = nn.Sequential(
            nn.Linear(gnn_hidden_dim, gnn_hidden_dim),
            nn.GELU(),
            nn.Linear(gnn_hidden_dim, cl_embed_dim)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr, batch_index):
        # A. Embed Atoms
        h = self.atom_encoder(x)

        # B. Run Local GNN (Integrate Bond Info)
        for conv, norm in zip(self.gnn_layers, self.norm_layers):
            h_in = h
            h = conv(h, edge_index, edge_attr=edge_attr)
            h = torch.relu(h)
            h = self.dropout(h)
            h = norm(h + h_in) # Residual

        # C. Convert to Dense Sequence for Transformer
        # dense_h: [Batch_Size, Max_Atoms, Hidden]
        # mask:    [Batch_Size, Max_Atoms] (True = Real Atom, False = Padding)
        dense_h, mask = to_dense_batch(h, batch_index)

        batch_size = dense_h.size(0)

        # D. Append CLS Token
        # We add a learnable token at the start of every sequence to summarize the molecule
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        dense_h = torch.cat((cls_tokens, dense_h), dim=1) # [B, L+1, H]

        # Adjust mask for CLS token (always True)
        cls_mask = torch.ones((batch_size, 1), device=mask.device, dtype=torch.bool)
        transformer_mask = torch.cat((cls_mask, mask), dim=1)

        # E. Run Transformer
        # src_key_padding_mask expects True for PADDING, so we invert our mask
        # ~transformer_mask means False for Real Atoms, True for Padding
        h_trans = self.transformer_encoder(
            dense_h,
            src_key_padding_mask=~transformer_mask
        )

        # F. Extract Outputs

        # 1. Sequence Output (for T5 Generation)
        # Project to T5 Dim (e.g. 256 -> 768)
        sequence_out = self.project_nodes(h_trans)

        # 2. Contrastive Embedding (from CLS token only)
        cls_output = h_trans[:, 0, :] # Take first token
        graph_cl_embed = self.cl_projector(cls_output)

        # Return: Sequence, Mask, CL Vector
        return sequence_out, transformer_mask.long(), graph_cl_embed


class TextDecoder(nn.Module):
    def __init__(self, model_name='t5-base', cl_embed_dim=256):
        super().__init__()
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)
        self.t5 = T5ForConditionalGeneration.from_pretrained(model_name)
        self.hidden_dim = self.t5.config.d_model

        # We reuse the T5 Encoder to get text embeddings, then project them
        self.cl_projector = nn.Sequential(
            nn.Linear(self.hidden_dim, self.hidden_dim),
            nn.GELU(),
            nn.Linear(self.hidden_dim, cl_embed_dim)
        )

    def forward(self, encoder_hidden_states, encoder_mask, labels=None):
        encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)
        return self.t5(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_mask,
            labels=labels
        ).loss

    def get_text_features(self, input_ids, attention_mask):
        """
        Extracts dense text features using the T5 Encoder.
        Useful for Contrastive Learning.
        """
        # We use the .encoder part of T5 (already pre-trained!)
        encoder_outputs = self.t5.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        last_hidden_state = encoder_outputs.last_hidden_state # [Batch, Seq, 768]

        # Mean Pooling (ignore padding)
        # Create mask for division (avoid div by zero)
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

        pooled_output = sum_embeddings / sum_mask # [Batch, 768]

        # Project to CL space
        return self.cl_projector(pooled_output) # [Batch, 256]

    def generate(self, encoder_hidden_states, encoder_mask, max_length=128, **kwargs):
        encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)
        return self.t5.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_mask,
            max_length=max_length,
            num_beams=4,
            early_stopping=True,
            decoder_start_token_id=self.t5.config.decoder_start_token_id,
            **kwargs
        )



class MoleculeCaptioningModel(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        # We re-use the corrected loss class
        self.loss_fn_cl = InfoNCELoss(temperature=0.07, learnable=True)

        # Learnable Uncertainty Weights
        self.s_gen = nn.Parameter(torch.zeros(1))
        self.s_cl = nn.Parameter(torch.zeros(1))

    def forward(self, x, edge_index, edge_attr, batch_index, labels, label_mask):
        # Training forward pass
        graph_seq_out, graph_mask, graph_cl_vec = self.encoder(x, edge_index, edge_attr, batch_index)

        gen_loss = self.decoder(graph_seq_out, graph_mask, labels=labels)

        clean_ids = labels.clone()
        clean_ids[clean_ids == -100] = self.decoder.tokenizer.pad_token_id
        text_cl_vec = self.decoder.get_text_features(clean_ids, label_mask)

        cl_loss = self.loss_fn_cl(graph_cl_vec, text_cl_vec)

        loss_gen_w = (gen_loss * torch.exp(-self.s_gen)) + (self.s_gen * 0.5)
        loss_cl_w  = (cl_loss  * torch.exp(-self.s_cl))  + (self.s_cl * 0.5)

        return loss_gen_w + loss_cl_w, gen_loss, cl_loss

    def predict(self, x, edge_index, edge_attr, batch_index):
        # 1. Run Encoder
        graph_seq_out, graph_mask, _ = self.encoder(x, edge_index, edge_attr, batch_index)

        # 2. Generate
        generated_ids = self.decoder.generate(graph_seq_out, graph_mask)

        # 3. Decode to Text
        return self.decoder.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

# ==========================================
# 3. SETUP & TRAINING LOOP
# ==========================================

# --- A. Prepare Data ---
tokenizer = T5Tokenizer.from_pretrained('t5-base')

print("Step 1: Tokenizing Dataset...")
all_data = train_graphs + val_graphs # Placeholder variable names

for graph in tqdm(all_data, desc="Tokenizing"):
    tokenized = tokenizer(
        graph.description,
        padding='max_length',
        truncation=True,
        max_length=CONFIG['max_len'],
        return_tensors="pt"
    )
    graph.y_ids = tokenized.input_ids.squeeze(0)

train_loader = DataLoader(train_graphs + val_graphs, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_graphs, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=4, pin_memory=True)

# --- B. Initialize Models ---
print("Step 2: Initializing Models...")
encoder = HybridGraphEncoder(
    num_edge_features=3,
    gnn_hidden_dim=256,
    output_dim=768,
    gnn_layers=2,
    transformer_layers=4,
    cl_embed_dim=CONFIG['cl_dim']
)
decoder = TextDecoder('t5-base', cl_embed_dim=CONFIG['cl_dim'])
model = MoleculeCaptioningModel(encoder, decoder).to(DEVICE)

# --- C. Load Weights (Smart Resume with Shape Checking) ---
if os.path.exists(CONFIG['prev_model_path']):
    print(f"Step 3: Loading weights from {CONFIG['prev_model_path']}...")

    # Load the checkpoint
    checkpoint = torch.load(CONFIG['prev_model_path'], map_location=DEVICE, weights_only=False)

    # Handle dictionary format
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
        print(f"   -> Found checkpoint from epoch {checkpoint.get('epoch', '?')}")
    else:
        state_dict = checkpoint

    # Get the current model's state dict to compare shapes
    model_state = model.state_dict()
    filtered_state_dict = {}
    skipped_layers = []

    for key, value in state_dict.items():
        if key in model_state:
            # CHECK: Do the shapes match?
            if value.shape == model_state[key].shape:
                filtered_state_dict[key] = value
            else:
                skipped_layers.append(key)
        # If key is not in model, strict=False handles it automatically, so we ignore here

    # Load only the matching layers
    model.load_state_dict(filtered_state_dict, strict=False)

    print(f"✅ Loaded matching weights successfully.")
    if len(skipped_layers) > 0:
        print(f"⚠️  Skipped {len(skipped_layers)} layers due to shape mismatch (Architecture changed).")
        print(f"    Examples: {skipped_layers[:3]} ...")
        print("    These layers (mostly the Encoder) will be re-initialized and trained from scratch.")
else:
    print(f"⚠️ Checkpoint not found at {CONFIG['prev_model_path']}. Training from scratch.")

# --- D. Optimizer ---
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'])
total_steps = len(train_loader) * CONFIG['epochs']
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps
)

# --- E. Main Loop ---
print(f"Step 4: Starting Training on {DEVICE} (Bfloat16 enabled)...")
best_val_loss = float('inf')

for epoch in range(CONFIG['epochs']):
    # --- TRAIN ---
    model.train()
    total_loss, total_gen, total_cl = 0, 0, 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}", leave=False)

    for batch in progress:
        batch = batch.to(DEVICE)

        # Prepare Labels
        text_inputs = batch.y_ids.view(batch.num_graphs, -1).to(DEVICE)
        text_mask = (text_inputs != tokenizer.pad_token_id).long()
        labels = text_inputs.clone()
        labels[labels == tokenizer.pad_token_id] = -100

        optimizer.zero_grad()

        # Mixed Precision (BFloat16 for A100)
        with autocast(dtype=torch.bfloat16):
            loss, gen_loss, cl_loss = model(
                x=batch.x,
                edge_index=batch.edge_index,
                edge_attr=batch.edge_attr.float(),
                batch_index=batch.batch,
                labels=labels,
                label_mask=text_mask
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        total_gen += gen_loss.item()
        total_cl += cl_loss.item()

        progress.set_postfix({
            "Loss": f"{loss.item():.2f}",
            "Gen": f"{gen_loss.item():.2f}",
            "CL": f"{cl_loss.item():.2f}"
        })

    avg_train = total_loss / len(train_loader)

    # --- VALIDATE ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            text_inputs = batch.y_ids.view(batch.num_graphs, -1).to(DEVICE)
            text_mask = (text_inputs != tokenizer.pad_token_id).long()
            labels = text_inputs.clone()
            labels[labels == tokenizer.pad_token_id] = -100

            with autocast(dtype=torch.bfloat16):
                v_loss, _, _ = model(
                    x=batch.x,
                    edge_index=batch.edge_index,
                    edge_attr=batch.edge_attr.float(),
                    batch_index=batch.batch,
                    labels=labels,
                    label_mask=text_mask
                )
            val_loss += v_loss.item()

    avg_val = val_loss / len(val_loader)

    # --- LOG & SAVE ---
    # Retrieve current learned weights for logging
    w_gen = torch.exp(-model.s_gen).item()
    w_cl = torch.exp(-model.s_cl).item()

    print(f"Epoch {epoch+1} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | Weights -> Gen: {w_gen:.2f}, CL: {w_cl:.2f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_val_loss,
            'config': CONFIG
        }, CONFIG['save_path'])
        print(f"  --> Saved Best Model to {CONFIG['save_path']}")

print("Done!")

# inference

In [ ]:
import torch
import pandas as pd
from torch_geometric.loader import DataLoader
from tqdm.notebook import tqdm
import os

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = base_drive_path / 'best_model_cl_finetuned.pth'
OUTPUT_CSV = base_drive_path / 'submission.csv'
BATCH_SIZE = 32
CL_EMBED_DIM = 256

print(f"Loading model from: {MODEL_PATH}")

# ==========================================
# 2. INITIALIZE MODEL ARCHITECTURE
# ==========================================
# 1. FIX: Added cl_embed_dim to match training
decoder = TextDecoder('t5-base', cl_embed_dim=CL_EMBED_DIM)

# In Step 2 of your main script:

encoder = HybridGraphEncoder(
    num_edge_features=3,
    gnn_hidden_dim=256,
    output_dim=768,
    gnn_layers=2,
    transformer_layers=4,
    cl_embed_dim=CONFIG['cl_dim']
)
decoder = TextDecoder('t5-base', cl_embed_dim=CONFIG['cl_dim'])
model = MoleculeCaptioningModel(encoder, decoder).to(DEVICE)

model = MoleculeCaptioningModel(encoder, decoder).to(DEVICE)

# ==========================================
# 3. LOAD TRAINED WEIGHTS (FIXED)
# ==========================================
if hasattr(torch.cuda, 'is_available') and torch.cuda.is_available():
    map_location = None
else:
    map_location = torch.device('cpu')

try:
    print(f"🔄 Attempting to load: {MODEL_PATH}")

    checkpoint = torch.load(MODEL_PATH, map_location=map_location, weights_only=False)

    # Handle Dictionary format
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        print("✅ Detected checkpoint dictionary. Extracting weights...")
        state_dict = checkpoint['model_state_dict']
    else:
        state_dict = checkpoint

    # Load weights
    model.load_state_dict(state_dict, strict=False)
    print("✅ Weights loaded successfully!")

except Exception as e:
    print(f"\n❌ CRITICAL ERROR: Could not load weights.")
    print(f"Error details: {e}")
    print("Stopping execution to prevent generating junk predictions.")
    raise e

# ==========================================
# 4. RUN INFERENCE ON TEST SET
# ==========================================
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model.eval()
predictions = []
ids = []

print(f"Generating captions for {len(test_graphs)} molecules...")

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference"):
        batch = batch.to(DEVICE)

        generated_text = model.predict(
            x=batch.x,
            edge_index=batch.edge_index,
            edge_attr=batch.edge_attr.float(),
            batch_index=batch.batch,
            # --- HYPERPARAMETERS ---
            num_beams=5,
            max_length=128,
            repetition_penalty=1.0,
            early_stopping=True
        )


        predictions.extend(generated_text)

        if isinstance(batch.id, list):
            ids.extend(batch.id)
        else:
            ids.extend(batch.id.tolist())

# ==========================================
# 5. SAVE SUBMISSION FILE
# ==========================================
df = pd.DataFrame({
    'ID': ids,
    'description': predictions
})

df.to_csv(OUTPUT_CSV, index=False)

print("------------------------------------------------")
print(f"🎉 Done! Submission saved to: {OUTPUT_CSV}")
print("First 3 predictions:")
print(df.head(3))
print("------------------------------------------------")

In [ ]:
df.to_csv("sub_cl_generated.csv")

In [ ]:
df